In [ ]:
import numpy as np
import pandas as pd
import MDAnalysis as mda
import matplotlib.pyplot as plt
#import plotly.express as px
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pickle

# 1. Define the model function
def constrained_exp(t, A, T1, c):
    T2 = c * T1
    return A * np.exp(-t / T1) + (1 - A) * np.exp(-t / T2)

# 2. Refined fit function to return Teff
def fit_constrained(sp, label):
    t = np.arange(len(sp))
    y = np.array(sp)
    
    # normalize
    y = y / y[0]
    
    # initial guesses and bounds
    p0 = [0.5, 10.0, 5.0]  # A, T1, c
    bounds = ([0, 0, 1], [1, np.inf, np.inf])
    
    popt, _ = curve_fit(constrained_exp, t, y, p0=p0, bounds=bounds)
    
    A, T1, c = popt
    T2 = c * T1
    tau_avg = A * T1 + (1 - A) * T2
    
    # Plotting
    plt.plot(t, y, 'o', alpha=0.3, label=f'{label} data')
    plt.plot(t, constrained_exp(t, *popt), '-', linewidth=2, label=f'{label} fit')
    
    return tau_avg


import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# triexponential model
def constrained_exp3(t, A1, A2, T1, T2, T3):
    A3 = 1 - A1 - A2
    return A1 * np.exp(-t / T1) + A2 * np.exp(-t / T2) + A3 * np.exp(-t / T3)


def fit_constrained3(sp, label):
    t = np.arange(len(sp))
    y = np.array(sp)

    # normalize
    y = y / y[0]

    # initial guesses
    p0 = [0.3, 0.3, 5.0, 20.0, 100.0]  # A1, A2, T1, T2, T3

    # bounds:
    # A1, A2 between 0 and 1
    # T1, T2, T3 > 0
    bounds = ([0, 0, 0, 0, 0],
              [1, 1, np.inf, np.inf, np.inf])

    popt, _ = curve_fit(constrained_exp3, t, y, p0=p0, bounds=bounds)

    A1, A2, T1, T2, T3 = popt
    A3 = 1 - A1 - A2

    # weighted average lifetime
    tau_avg = A1*T1 + A2*T2 + A3*T3

    # Plotting
    plt.plot(t, y, 'o', alpha=0.3, label=f'{label} data')
    plt.plot(t, constrained_exp3(t, *popt), '-', linewidth=2, label=f'{label} fit')

    print(f"{label}:")
    print(f"A1={A1:.3f}, T1={T1:.2f}")
    print(f"A2={A2:.3f}, T2={T2:.2f}")
    print(f"A3={A3:.3f}, T3={T3:.2f}")
    print(f"tau_avg={tau_avg:.2f}")

    return tau_avg



from scipy.optimize import curve_fit
import numpy as np

# tri-exponential
def exp3(t, A1, A2, T1, T2, T3):
    A3 = 1 - A1 - A2
    return (A1*np.exp(-t/T1) +
            A2*np.exp(-t/T2) +
            A3*np.exp(-t/T3))

def fit_exp3(t, y):
    # remove nan/inf
    mask = np.isfinite(t) & np.isfinite(y)
    t = np.array(t)[mask]
    y = np.array(y)[mask]

    # normalize
    y = y / y[0]

    # initial guess
    p0 = [0.3, 0.3, 5, 50, 500]

    bounds = ([0, 0, 0, 0, 0],
              [1, 1, np.inf, np.inf, np.inf])

    popt, _ = curve_fit(exp3, t, y, p0=p0, bounds=bounds, maxfev=10000)

    A1, A2, T1, T2, T3 = popt
    A3 = 1 - A1 - A2
    tau_avg = A1*T1 + A2*T2 + A3*T3

    return popt, tau_avg

## Taking whole systems

In [ ]:
datasets = [
    ('corr_results/all_water_corr_result_3res.pkl',
     ['equal', 'aromatic', 'nonpolar', 'polar', 'charged'],
     '3res'),

    ('corr_results/all_water_corr_result_9res.pkl',
     ['equal', 'aromatic', 'nonpolar', 'polar', 'charged'],
     '9res'),

    ('corr_results/all_water_corr_result_9res_2.pkl',
     ['2_close','2_spread','3_close','3_spread'],
     '9res (spacers)'),
]

# Colors
clrs = ['black','red','blue','green','purple','orange']

plt.rcParams.update({"font.size": 14})

# ── Create subplot grid (3 rows × 2 cols) ───────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(12, 12))

# ── Loop ───────────────────────────────────────────────────────────
for row, (save_path, systems, title_row) in enumerate(datasets):

    with open(save_path, 'rb') as f:
        results = pickle.load(f)

    ax_corr = axes[row, 0]
    ax_bar  = axes[row, 1]

    tau_vals = []

    # ── Correlation plots ──────────────────────────────────────────
    for i, label in enumerate(systems):
        r = results[label]

        ax_corr.plot(r['tM'][::5], r['corrN'][::5], 'o', color=clrs[i], alpha=0.2)
        
        # fit
        try:
            popt, tau_avg = fit_exp3(r['tM'], r['corrN'])
            yfit = exp3(r['tM'], *popt)

            ax_corr.plot(r['tM'], yfit, '-', color=clrs[i], lw=2,
                         label=f'{label} ({tau_avg:.3f})')

            tau_vals.append(tau_avg)

        except RuntimeError:
            print(f"Fit failed for {label}")
            tau_vals.append(np.nan)
            
            
    ax_corr.set_title(f"{title_row} — Correlation")
    ax_corr.set_xlabel("t [ns]")
    ax_corr.set_ylabel("C(t)")
    ax_corr.legend(frameon=False, fontsize=10)
    

    # ── Bar plot ───────────────────────────────────────────────────
    x = np.arange(len(systems))
    ax_bar.bar(x, tau_vals, color=clrs[:len(systems)])

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(systems, rotation=45)
    ax_bar.set_ylabel("τ_eff [ns]")
    ax_bar.set_title(f"{title_row} — τ comparison")

# ── Layout ─────────────────────────────────────────────────────────
plt.tight_layout()
plt.show()            

        

### Survial Probability

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ── Model ───────────────────────────────────────────────────────────
def exp3(t, A1, A2, T1, T2, T3):
    A3 = 1 - A1 - A2
    return (A1*np.exp(-t/T1) +
            A2*np.exp(-t/T2) +
            A3*np.exp(-t/T3))


def fit_exp3(t, y):
    y = np.array(y)
    t = np.array(t)

    mask = np.isfinite(y)
    t = t[mask]
    y = y[mask]

    y = y / y[0]

    p0 = [0.3, 0.3, 5, 50, 500]
    bounds = ([0, 0, 0, 0, 0],
              [1, 1, np.inf, np.inf, np.inf])

    popt, _ = curve_fit(exp3, t, y, p0=p0, bounds=bounds, maxfev=10000)

    A1, A2, T1, T2, T3 = popt
    A3 = 1 - A1 - A2

    tau_eff = A1*T1 + A2*T2 + A3*T3

    return tau_eff


# ── Systems ────────────────────────────────────────────────────────
systems_3res = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
systems_9res = systems_3res
systems_spacers = ['2_close','2_spread','3_close','3_spread']


# ── Collect τeff ───────────────────────────────────────────────────
teff_3res = {}
for sys in systems_3res:
    df = pd.read_csv(f'data/sp_all_3res_{sys}_new.csv')

    t = np.arange(len(df['SP_all']))
    teff_3res[sys] = fit_exp3(t, df['SP_all'])


teff_9res = {}
for sys in systems_9res:
    df = pd.read_csv(f'data/sp_all_9res_{sys}_new.csv')

    t = np.arange(len(df['SP_all']))
    teff_9res[sys] = fit_exp3(t, df['SP_all'])


teff_spacers = {}
for sys in systems_spacers:
    df = pd.read_csv(f'data/sp_all_9res_{sys}_new.csv')

    t = np.arange(len(df['SP_all']))
    teff_spacers[sys] = fit_exp3(t, df['SP_all'])


# ── Plot setup ─────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(10, 12))

def plot_group(ax, data, title):
    systems = list(data.keys())
    vals = list(data.values())

    x = np.arange(len(systems))

    ax.bar(x, vals, color='steelblue')
    ax.set_xticks(x)
    ax.set_xticklabels(systems, rotation=45)
    ax.set_ylabel("τ_eff")
    ax.set_title(title)


# ── Plots ──────────────────────────────────────────────────────────
plot_group(axes[0], teff_3res, "3-res Systems")
plot_group(axes[1], teff_9res, "9-res Systems")
plot_group(axes[2], teff_spacers, "Spacer Systems")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ─────────────────────────────────────────────────────────────
# TRI-EXPONENTIAL MODEL
# ─────────────────────────────────────────────────────────────
def exp3(t, A1, A2, T1, T2, T3):
    A3 = 1 - A1 - A2
    return (A1*np.exp(-t/T1) +
            A2*np.exp(-t/T2) +
            A3*np.exp(-t/T3))


# ─────────────────────────────────────────────────────────────
# FIT FUNCTION
# ─────────────────────────────────────────────────────────────
def fit_exp3(t, y):

    t = np.array(t)
    y = np.array(y)

    mask = np.isfinite(y)
    t = t[mask]
    y = y[mask]

    # normalize
    y = y / y[0]

    # initial guesses
    p0 = [0.3, 0.3, 5, 50, 500]

    bounds = ([0, 0, 0, 0, 0],
              [1, 1, np.inf, np.inf, np.inf])

    popt, _ = curve_fit(exp3, t, y,
                        p0=p0,
                        bounds=bounds,
                        maxfev=20000)

    A1, A2, T1, T2, T3 = popt
    A3 = 1 - A1 - A2

    tau_eff = A1*T1 + A2*T2 + A3*T3

    return popt, tau_eff


# ─────────────────────────────────────────────────────────────
# SYSTEMS
# ─────────────────────────────────────────────────────────────
systems_3res = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
systems_9res = systems_3res
systems_spacers = ['2_close','2_spread','3_close','3_spread']


# ─────────────────────────────────────────────────────────────
# PLOT SETUP
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3,2 , figsize=(14, 12))


# ─────────────────────────────────────────────────────────────
# MAIN FUNCTION
# ─────────────────────────────────────────────────────────────
def process(file_prefix, systems, ax_decay, ax_bar, title):

    tau_vals = []

    for sys in systems:

        df = pd.read_csv(f"data/{file_prefix}_{sys}_new.csv")

        t = np.arange(len(df["SP_all"]))
        y = np.array(df["SP_all"])

        # normalize
        y = y / y[0]

        # fit
        popt, tau_eff = fit_exp3(t, y)
        yfit = exp3(t, *popt)

        # ── DECAY PLOT ─────────────────────────────
        ax_decay.plot(t[::8], y[::8], 'o', alpha=0.25)
        ax_decay.plot(t, yfit, lw=2, label=sys)

        tau_vals.append(tau_eff)

    ax_decay.set_title(f"{title} — Decay")
    ax_decay.set_xlabel("t")
    ax_decay.set_ylabel("C(t)")
    ax_decay.legend(frameon=False, fontsize=8)
    ax_decay.set_ylim(0,0.4)

    # ── BAR PLOT ────────────────────────────────
    x = np.arange(len(systems))
    ax_bar.bar(x, tau_vals, color="steelblue")

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(systems, rotation=45)
    ax_bar.set_ylabel("τ_eff")
    ax_bar.set_title(f"{title} — Relaxation time")
    
    ax_bar.set_ylim(10,60)


# ─────────────────────────────────────────────────────────────
# RUN ALL SYSTEMS
# ─────────────────────────────────────────────────────────────
process('sp_all_3res', systems_3res,
        axes[0,0], axes[0,1],
        "3-res systems")

process('sp_all_9res', systems_9res,
        axes[1,0], axes[1,1],
        "9-res systems")

process('sp_all_9res', systems_spacers,
        axes[2,0], axes[2,1],
        "Spacer systems")


# ─────────────────────────────────────────────────────────────
# FINALIZE
# ─────────────────────────────────────────────────────────────
plt.tight_layout()
plt.show()

### inner and outer layer 


In [ ]:
# ─────────────────────────────────────────────────────────────
# TRI-EXPONENTIAL MODEL
# ─────────────────────────────────────────────────────────────
def exp3(t, A1, A2, T1, T2, T3):
    A3 = 1 - A1 - A2
    return (A1*np.exp(-t/T1) +
            A2*np.exp(-t/T2) +
            A3*np.exp(-t/T3))


# ─────────────────────────────────────────────────────────────
# FIT FUNCTION
# ─────────────────────────────────────────────────────────────
def fit_exp3terms(t, y):

    t = np.array(t)
    y = np.array(y)

    mask = np.isfinite(y)
    t = t[mask]
    y = y[mask]

    # normalize
    y = y / y[0]

    # initial guesses
    p0 = [0.3, 0.3, 5, 50, 500]

    bounds = ([0, 0, 0, 0, 0],
              [1, 1, np.inf, np.inf, np.inf])

    popt, _ = curve_fit(exp3, t, y,
                        p0=p0,
                        bounds=bounds,
                        maxfev=20000)

    A1, A2, T1, T2, T3 = popt
    A3 = 1 - A1 - A2

    tau_eff = A1*T1 + A2*T2 + A3*T3
    
    yfit = exp3(t, *popt)

    return popt, tau_eff,yfit




import pickle
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [ ]:


# ── Config ─────────────────────────────────────────────────────────
pkl_files = [
    ('./corr_results/water_corr_resultsi_3res.pkl',   'Dataset 1 (3res)'),
    ('./corr_results/water_corr_resultsi_9res.pkl',   'Dataset 2 (9res)'),
    ('./corr_results/water_corr_resultsi_9res_2.pkl', 'Dataset 3 (9res_2)'),
]

clrs  = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2 = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

row_titles = ['3res', '9res', '9res_2']

# ── Figure ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 15))


# ── LOOP OVER DATASETS ────────────────────────────────────────────
for row, (pkl_path, _) in enumerate(pkl_files):

    with open(pkl_path, 'rb') as f:
        results = pickle.load(f)

    ax_in  = axes[row, 0]
    ax_out = axes[row, 1]

    ax_in.set_title(f'Inner Layer — {row_titles[row]}',
                    fontsize=12, fontweight='bold')
    ax_out.set_title(f'Outer Layer — {row_titles[row]}',
                     fontsize=12, fontweight='bold')

    labels = list(results.keys())

    tau_inner = []
    tau_outer = []

    # ── CURVES + FITS ──────────────────────────────────────────────
    for idx, label in enumerate(labels):
        res = results[label]
        tM = np.array(res['tM'])

        # ── INNER ───────────────────────────────────────────────────
        y_in = np.array(res['corrN'])

        popt_in, tau_in, yfit_in = fit_exp3terms(tM, y_in)

        ax_in.plot(tM, y_in, color=clrs[idx], lw=1.8, alpha=0.6)
        ax_in.plot(tM, yfit_in, color=clrs[idx], lw=2,
                   label=f"{label} (τ={tau_in:.2f})")

        tau_inner.append(tau_in)

        # ── OUTER ───────────────────────────────────────────────────
        y_out = np.array(res['corrN_2'])
        ystd  = np.array(res['corrN_std_2'])

        popt_out, tau_out, yfit_out = fit_exp3terms(tM, y_out)

        ax_out.fill_between(tM,
                            y_out - ystd,
                            y_out + ystd,
                            color=clrs2[idx], alpha=0.15)

        ax_out.plot(tM, y_out, color=clrs2[idx], lw=1.8, alpha=0.6)
        ax_out.plot(tM, yfit_out, color=clrs2[idx], lw=2,
                    label=f"{label} (τ={tau_out:.2f})")
        
        

        tau_outer.append(tau_out)

    # ── AXES STYLE ────────────────────────────────────────────────
    for ax in (ax_in, ax_out):
        ax.set_xlabel('Time (ns)', fontsize=11)
        ax.set_ylabel('C(t)', fontsize=11)
        ax.set_xlim(tM[0], tM[-1])
        ax.set_ylim(-0.05, 0.5)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.tick_params(labelsize=9)

    # ── τ INSET PLOTS ─────────────────────────────────────────────
    x = np.arange(len(labels))

    # INNER
    axins_in = inset_axes(ax_in, width="45%", height="45%", loc='upper center')
    axins_in.bar(x, tau_inner, color=clrs[:len(labels)], alpha=0.9)
    axins_in.set_xticks([])
    axins_in.tick_params(axis='y', labelsize=7)
    axins_in.grid(True, alpha=0.2)

    for i, v in enumerate(tau_inner):
        axins_in.text(i, v, f"{v:.2f}", ha='center', va='bottom', fontsize=6)

    # OUTER
    axins_out = inset_axes(ax_out, width="45%", height="45%", loc='upper center')
    axins_out.bar(x, tau_outer, color=clrs2[:len(labels)], alpha=0.9)
    axins_out.set_xticks([])
    axins_out.tick_params(axis='y', labelsize=7)
    axins_out.grid(True, alpha=0.2)

    for i, v in enumerate(tau_outer):
        axins_out.text(i, v, f"{v:.2f}", ha='center', va='bottom', fontsize=6)

# ── FINAL ─────────────────────────────────────────────────────────
plt.suptitle('Water–Protein Contact Correlation (Tri-exponential Fit)',
             fontsize=15, fontweight='bold', y=1.01)

plt.tight_layout()

save_path = './corr_results/water_corr_comparison_3x2_triexp.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

print(f"Saved: {save_path}")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────
# SYSTEMS
# ─────────────────────────────────────────────────────────────
systems_3res = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
systems_9res = systems_3res
systems_spacers = ['2_close','2_spread','3_close','3_spread']

layers = ['Inner', 'Outer']


# ─────────────────────────────────────────────────────────────
# COMPUTE TEFF (INNER + OUTER)
# ─────────────────────────────────────────────────────────────
def compute_teff(df):
    out = {}

    t = np.arange(len(df[layers[0]]))

    for lay in layers:
        y = np.array(df[lay])

        popt, tau_eff, _ = fit_exp3terms(t, y)

        out[lay] = tau_eff

    return out


# ─────────────────────────────────────────────────────────────
# COLLECT DATA
# ─────────────────────────────────────────────────────────────
teff_3res = {}
for sys in systems_3res:
    df = pd.read_csv(f'data/sp_2res_{sys}_new.csv')
    teff_3res[sys] = compute_teff(df)



teff_9res = {}
for sys in systems_9res:
    df = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
    teff_9res[sys] = compute_teff(df)


teff_spacers = {}
for sys in systems_spacers:
    df = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
    teff_spacers[sys] = compute_teff(df)


# ─────────────────────────────────────────────────────────────
# PLOTTING FUNCTION
# ─────────────────────────────────────────────────────────────
def plot_group(ax, data, title):

    systems = list(data.keys())
    x_base = np.arange(len(layers))   # Inner, Outer

    width = 0.15

    for i, sys in enumerate(systems):

        vals = [data[sys]['Inner'], data[sys]['Outer']]

        ax.bar(x_base + i*width, vals, width=width, label=sys)

    ax.set_xticks(x_base + width*(len(systems)-1)/2)
    ax.set_xticklabels(layers)
    ax.set_ylabel("τ_eff (ns)")
    ax.set_title(title)
    ax.legend(fontsize=8)

# ─────────────────────────────────────────────────────────────
# FIGURE
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

plot_group(axes[0], teff_3res, "3-res Systems (3-term fit)")
plot_group(axes[1], teff_9res, "9-res Systems (3-term fit)")
plot_group(axes[2], teff_spacers, "Spacer Systems (3-term fit)")

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# COMPUTE + STORE FITS
# ─────────────────────────────────────────────────────────────
def process_systems(file_prefix, systems):

    results = {}

    for sys in systems:

        df = pd.read_csv(f'data/{file_prefix}_{sys}_new.csv')

        t = np.arange(len(df['Inner']))  # same time for both layers

        sys_out = {}

        for lay in layers:

            y = np.array(df[lay])
            y = y / y[0]   # normalize

            popt, tau_eff, yfit = fit_exp3terms(t, y)

            sys_out[lay] = {
                "tau": tau_eff,
                "y": y,
                "fit": yfit
            }

        results[sys] = sys_out

    return results


# ─────────────────────────────────────────────────────────────
# LOAD ALL DATASETS
# ─────────────────────────────────────────────────────────────
data_3res = process_systems('sp_2res', systems_3res)
data_9res = process_systems('sp_9res', systems_9res)
data_spacers = process_systems('sp_9res', systems_spacers)


# ─────────────────────────────────────────────────────────────
# PLOTTING FUNCTION
# ─────────────────────────────────────────────────────────────
def plot_all(fig_title, data, fig_idx):

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax_decay = axes[0]
    ax_bar = axes[1]

    colors = ['r','b','g','m','c']

    systems = list(data.keys())

    tau_inner = []
    tau_outer = []

    # ── DECAY PLOTS ─────────────────────────────────────────────
    for i, sys in enumerate(systems):

        t = None

        for j, lay in enumerate(layers):

            y = data[sys][lay]['y']
            yfit = data[sys][lay]['fit']
            tau = data[sys][lay]['tau']

            if t is None:
                t = np.arange(len(y))

            if lay == 'Inner':
                ax_decay.plot(t[::5], y[::5], 'o', color=colors[i], alpha=0.3)
                ax_decay.plot(t, yfit, '-', color=colors[i],
                              label=f"{sys} Inner (τ={tau:.1f})")
                tau_inner.append(tau)

            else:
                ax_decay.plot(t, yfit, '--', color=colors[i],
                              label=f"{sys} Outer (τ={tau:.1f})")
                tau_outer.append(tau)

    ax_decay.set_title(f"{fig_title} — Decay + Fit")
    ax_decay.set_xlabel("Time")
    ax_decay.set_ylabel("Survival Probability")
    ax_decay.set_ylim(-0.05, 1.05)
    ax_decay.legend(fontsize=8)
    ax_decay.grid(alpha=0.3)

    # ── BAR PLOT ───────────────────────────────────────────────
    x = np.arange(len(systems))
    w = 0.35

    ax_bar.bar(x - w/2, tau_inner, width=w, label='Inner')
    ax_bar.bar(x + w/2, tau_outer, width=w, label='Outer')

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(systems, rotation=45)
    ax_bar.set_ylabel("τ_eff")
    ax_bar.set_title("Relaxation Times")
    ax_bar.legend()

    plt.tight_layout()
    return fig


# ─────────────────────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────────────────────
plot_all("3-res Systems", data_3res, 0)
plot_all("9-res Systems", data_9res, 1)
plot_all("Spacer Systems", data_spacers, 2)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────
# COMPUTE + STORE FITS
# ─────────────────────────────────────────────────────────────
def process_systems(file_prefix, systems):

    results = {}

    for sys in systems:

        df = pd.read_csv(f'data/{file_prefix}_{sys}_new.csv')

        t = np.arange(len(df['Inner']))  # same time for both layers

        sys_out = {}

        for lay in layers:

            y = np.array(df[lay])
            y = y / y[0]   # normalize

            popt, tau_eff, yfit = fit_exp3terms(t, y)

            sys_out[lay] = {
                "tau": tau_eff,
                "y": y,
                "fit": yfit
            }

        results[sys] = sys_out

    return results


# ─────────────────────────────────────────────────────────────
# LOAD ALL DATASETS
# ─────────────────────────────────────────────────────────────
data_3res = process_systems('sp_2res', systems_3res)
data_9res = process_systems('sp_9res', systems_9res)
data_spacers = process_systems('sp_9res', systems_spacers)


# ─────────────────────────────────────────────────────────────
# PLOTTING FUNCTION
# ─────────────────────────────────────────────────────────────
def plot_all(fig_title, data, fig_idx):

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax_decay = axes[0]
    ax_bar = axes[1]

    colors = ['r','b','g','m','c']
    systems = list(data.keys())

    tau_inner = []
    tau_outer = []

    # ── DECAY PLOTS ─────────────────────────────────────────────
    for i, sys in enumerate(systems):

        t = None

        for j, lay in enumerate(layers):

            y = data[sys][lay]['y']
            yfit = data[sys][lay]['fit']
            tau = data[sys][lay]['tau']

            if t is None:
                t = np.arange(len(y))

            if lay == 'Inner':
                ax_decay.plot(t[::5], y[::5], 'o', color=colors[i], alpha=0.3)
                ax_decay.plot(t, yfit, '-', color=colors[i],
                              label=f"{sys} Inner (τ={tau:.1f})")
                tau_inner.append(tau)

            else:
                ax_decay.plot(t, yfit, '--', color=colors[i],
                              label=f"{sys} Outer (τ={tau:.1f})")
                tau_outer.append(tau)

    ax_decay.set_title(f"{fig_title} — Decay + Fit")
    ax_decay.set_xlabel("Time")
    ax_decay.set_ylabel("Survival Probability")
    ax_decay.set_ylim(-0.05, 1.05)
    ax_decay.legend(fontsize=8)
    ax_decay.grid(alpha=0.3)

    # ── BAR PLOT (DUAL AXIS) ───────────────────────────────────
    x = np.arange(len(systems))
    w = 0.35

    # Left axis → Inner
    ax_bar.bar(x - w/2, tau_inner, width=w, label='Inner', color='tab:blue')
    ax_bar.set_ylabel("Inner τ_eff", color='tab:blue')
    ax_bar.tick_params(axis='y', labelcolor='tab:blue')

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(systems, rotation=45)

    # Right axis → Outer
    ax_bar2 = ax_bar.twinx()
    ax_bar2.bar(x + w/2, tau_outer, width=w, label='Outer', color='tab:red', alpha=0.7)
    ax_bar2.set_ylabel("Outer τ_eff", color='tab:red')
    ax_bar2.tick_params(axis='y', labelcolor='tab:red')

    # OPTIONAL: invert outer axis for “opposite sense”
    #ax_bar2.invert_yaxis()
    ax_bar2.set_ylim(0,18)
    # Title
    ax_bar.set_title("Relaxation Times (Inner vs Outer)")

    # Combined legend
    ax_bar.legend(loc='upper left')
    ax_bar2.legend(loc='upper right')

    plt.tight_layout()
    return fig


# ─────────────────────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────────────────────
plot_all("3-res Systems", data_3res, 0)
plot_all("9-res Systems", data_9res, 1)
plot_all("Spacer Systems", data_spacers, 2)

plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# PROCESS DATA
# ─────────────────────────────────────────────────────────────
def process(file_prefix, systems):

    results = {}

    for sys in systems:

        df = pd.read_csv(f'data/{file_prefix}_{sys}_new.csv')

        t = np.arange(len(df['Inner']))

        sys_dict = {}

        for lay in layers:

            y = np.array(df[lay])
            y = y / y[0]

            popt, tau_eff, yfit = fit_exp3terms(t, y)

            sys_dict[lay] = {
                "y": y,
                "fit": yfit,
                "tau": tau_eff
            }

        results[sys] = sys_dict

    return results


# ─────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────
data_3res = process('sp_2res', systems_3res)
data_9res = process('sp_9res', systems_9res)
data_spacers = process('sp_9res', systems_spacers)


# ─────────────────────────────────────────────────────────────
# PLOTTING FUNCTION
# ─────────────────────────────────────────────────────────────
def plot_system(title, data):

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax_in = axes[0]
    ax_out = axes[1]

    colors = ['r','b','g','m','c']

    systems = list(data.keys())

    tau_in, tau_out = [], []

    # ── INNER ───────────────────────────────────────────────
    for i, sys in enumerate(systems):

        t = np.arange(len(data[sys]['Inner']['y']))

        y = data[sys]['Inner']['y']
        yfit = data[sys]['Inner']['fit']

        ax_in.plot(t[::5], y[::5], 'o', color=colors[i], alpha=0.3)
        ax_in.plot(t, yfit, '-', color=colors[i],
                   label=f"{sys} (τ={data[sys]['Inner']['tau']:.1f})")

        tau_in.append(data[sys]['Inner']['tau'])

    ax_in.set_title(f"{title} — Inner")
    ax_in.set_xlabel("Time")
    ax_in.set_ylabel("Survival Probability")
    ax_in.set_ylim(-0.05, 1.05)
    ax_in.grid(alpha=0.3)
    ax_in.legend(fontsize=8)

    # ── OUTER ───────────────────────────────────────────────
    for i, sys in enumerate(systems):

        t = np.arange(len(data[sys]['Outer']['y']))

        y = data[sys]['Outer']['y']
        yfit = data[sys]['Outer']['fit']

        ax_out.plot(t[::5], y[::5], 'o', color=colors[i], alpha=0.3)
        ax_out.plot(t, yfit, '--', color=colors[i],
                    label=f"{sys} (τ={data[sys]['Outer']['tau']:.1f})")

        tau_out.append(data[sys]['Outer']['tau'])

    ax_out.set_title(f"{title} — Outer")
    ax_out.set_xlabel("Time")
    ax_out.set_ylabel("Survival Probability")
    ax_out.set_ylim(-0.05, 1.05)
    ax_out.grid(alpha=0.3)
    ax_out.legend(fontsize=8)

    plt.tight_layout()
    return fig


# ─────────────────────────────────────────────────────────────
# RUN PLOTS
# ─────────────────────────────────────────────────────────────
plot_system("3-res Systems", data_3res)
plot_system("9-res Systems", data_9res)
plot_system("Spacer Systems", data_spacers)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from scipy.optimize import curve_fit

# ── Tri-exponential model ──────────────────────────────────────────
def exp3(t, A1, A2, T1, T2, T3):
    A3 = 1 - A1 - A2
    return (A1*np.exp(-t/T1) +
            A2*np.exp(-t/T2) +
            A3*np.exp(-t/T3))

# ── Fit function ───────────────────────────────────────────────────
def fit_exp3(t, y):
    mask = np.isfinite(t) & np.isfinite(y)
    t = np.array(t)[mask]
    y = np.array(y)[mask]

    # normalize
    y = y / y[0]

    # initial guesses
    p0 = [0.3, 0.3, 5, 50, 500]

    bounds = ([0, 0, 0, 0, 0],
              [1, 1, np.inf, np.inf, np.inf])

    popt, _ = curve_fit(exp3, t, y, p0=p0, bounds=bounds, maxfev=10000)

    A1, A2, T1, T2, T3 = popt
    A3 = 1 - A1 - A2

    # weighted average lifetime
    tau_avg = A1*T1 + A2*T2 + A3*T3

    return popt, tau_avg, (T1, T2, T3)


# ── Datasets ───────────────────────────────────────────────────────
datasets = [
    ('corr_results/all_water_corr_result_3res.pkl',
     ['equal', 'aromatic', 'nonpolar', 'polar', 'charged'],
     '3res'),

    ('corr_results/all_water_corr_result_9res.pkl',
     ['equal', 'aromatic', 'nonpolar', 'polar', 'charged'],
     '9res'),

    ('corr_results/all_water_corr_result_9res_2.pkl',
     ['2_close','2_spread','3_close','3_spread'],
     '9res (spacers)'),
]

# Colors
clrs = ['black','red','blue','green','purple','orange']

plt.rcParams.update({"font.size": 14})

# ── Create subplot grid ────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

# ── Loop through datasets ──────────────────────────────────────────
for row, (save_path, systems, title_row) in enumerate(datasets):

    with open(save_path, 'rb') as f:
        results = pickle.load(f)

    ax_corr = axes[row, 0]
    ax_bar  = axes[row, 1]

    tau_vals = []
    T1_vals = []
    T2_vals = []
    T3_vals = []

    # ── Correlation plots ──────────────────────────────────────────
    for i, label in enumerate(systems):
        r = results[label]

        t = np.array(r['tM'])
        y = np.array(r['corrN'])

        # raw data
        ax_corr.plot(t[::5], y[::5], 'o',
                     color=clrs[i], alpha=0.2)

        # fit
        try:
            popt, tau_avg, Ts = fit_exp3(t, y)
            yfit = exp3(t, *popt)

            T1, T2, T3 = Ts

            ax_corr.plot(t, yfit, '-', color=clrs[i], lw=2,
                         label=f'{label} ({tau_avg:.1f})')

            tau_vals.append(tau_avg)
            T1_vals.append(T1)
            T2_vals.append(T2)
            T3_vals.append(T3)

        except RuntimeError:
            print(f"Fit failed for {label}")
            tau_vals.append(np.nan)
            T1_vals.append(np.nan)
            T2_vals.append(np.nan)
            T3_vals.append(np.nan)

    ax_corr.set_title(f"{title_row} — Correlation")
    ax_corr.set_xlabel("t [ns]")
    ax_corr.set_ylabel("C(t)")
    ax_corr.legend(frameon=False, fontsize=10)

    # Uncomment for semilog view
    # ax_corr.set_yscale('log')

    # ── Bar plot with twin y-axis ─────────────────────────────────────
    x = np.arange(len(systems))
    w = 0.2

    # left axis for slow scales
    ax_bar.bar(x - 0.5*w, T3_vals, width=w, label='T3')
    ax_bar.bar(x + 0.5*w, tau_vals, width=w, label='τeff')

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(systems, rotation=45)
    ax_bar.set_ylabel("Slow Timescale [ns]")
    ax_bar.set_title(f"{title_row} — Timescale comparison")

    # right axis for fast scales
    ax2 = ax_bar.twinx()
    ax2.bar(x - 1.5*w, T1_vals, width=w, alpha=0.7, label='T1')
    ax2.bar(x + 1.5*w, T2_vals, width=w, alpha=0.7, label='T2')

    ax2.set_ylabel("Fast Timescale [ns]")

    # combine legends
    handles1, labels1 = ax_bar.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax_bar.legend(handles1 + handles2, labels1 + labels2,
                  frameon=False, fontsize=9, loc='upper left')

# ── Layout ─────────────────────────────────────────────────────────
plt.tight_layout()
plt.show()

In [ ]:
## SASA vs cluster 


In [ ]:
j = 'equal'
df = pd.read_csv(f'data/{j}_cluster_water_9res.csv')
df.tail()

In [ ]:
# ── Files and systems ───────────────────────────────────────────────


datasets = [
    ('corr_results/all_water_corr_result_3res.pkl',
     ['equal', 'aromatic', 'nonpolar', 'polar', 'charged'],
     '3res'),

    ('corr_results/all_water_corr_result_9res.pkl',
     ['equal', 'aromatic', 'nonpolar', 'polar', 'charged'],
     '9res'),

    ('corr_results/all_water_corr_result_9res_2.pkl',
     ['2_close','2_spread','3_close','3_spread'],
     '9res (spacers)'),
]

# Colors
clrs = ['black','red','blue','green','purple','orange']

plt.rcParams.update({"font.size": 14})

# ── Create subplot grid (3 rows × 2 cols) ───────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(12, 12))

# ── Loop ───────────────────────────────────────────────────────────
for row, (save_path, systems, title_row) in enumerate(datasets):

    with open(save_path, 'rb') as f:
        results = pickle.load(f)

    ax_corr = axes[row, 0]
    ax_bar  = axes[row, 1]

    tau_vals = []

    # ── Correlation plots ──────────────────────────────────────────
    for i, label in enumerate(systems):
        r = results[label]

        ax_corr.plot(r['tM'], r['corrN'], '.', color=clrs[i], alpha=0.5)
        ax_corr.plot(r['tM'], r['corrFit'], '-', color=clrs[i],
                     label=f"{label} (τ={r['t_eff']:.2f})")
        
        ax_corr.set_ylim(0,0.4)

        tau_vals.append(r['t_eff'])

    ax_corr.set_title(f"{title_row} — Correlation")
    ax_corr.set_xlabel("t [ns]")
    ax_corr.set_ylabel("C(t)")
    ax_corr.legend(frameon=False, fontsize=10)
    

    # ── Bar plot ───────────────────────────────────────────────────
    x = np.arange(len(systems))
    ax_bar.bar(x, tau_vals, color=clrs[:len(systems)])

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(systems, rotation=45)
    ax_bar.set_ylabel("τ_eff [ns]")
    ax_bar.set_title(f"{title_row} — τ comparison")

# ── Layout ─────────────────────────────────────────────────────────
plt.tight_layout()
plt.show()



In [ ]:

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

datasets = [
    ('corr_results/all_water_corr_result_3res.pkl',
     ['equal', 'aromatic', 'nonpolar', 'polar', 'charged'],
     '3res'),
    ('corr_results/all_water_corr_result_9res.pkl',
     ['equal', 'aromatic', 'nonpolar', 'polar', 'charged'],
     '9res'),
    ('corr_results/all_water_corr_result_9res_2.pkl',
     ['2_close','2_spread','3_close','3_spread'],
     '9res (spacers)'),
]

clrs = ['black','red','blue','green','purple','orange']
plt.rcParams.update({"font.size": 14})

# ── Create subplot grid (1 row × 3 cols) ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Loop ─────────────────────────────────────────────────────────────
for col, (save_path, systems, title_row) in enumerate(datasets):
    with open(save_path, 'rb') as f:
        results = pickle.load(f)

    ax_corr = axes[col]
    tau_vals = []

    # ── Correlation plots ─────────────────────────────────────────────
    for i, label in enumerate(systems):
        r = results[label]
        ax_corr.plot(r['tM'], r['corrN'], '.', color=clrs[i], alpha=0.5)
        ax_corr.plot(r['tM'], r['corrFit'], '-', color=clrs[i],
                     label=f"{label} (τ={r['t_eff']:.2f})")
        tau_vals.append(r['t_eff'])

    ax_corr.set_ylim(0, 0.4)
    ax_corr.set_title(f"{title_row}")
    ax_corr.set_xlabel("t [ns]")
    ax_corr.set_ylabel("C(t)")
    ax_corr.legend(frameon=False, fontsize=9)

    # ── Inset bar plot ────────────────────────────────────────────────
    ax_inset = inset_axes(ax_corr,
                          width="55%", height="45%",
                          loc="upper right")      # tweak loc as needed

    x = np.arange(len(systems))
    ax_inset.bar(x, tau_vals, color=clrs[:len(systems)])
    ax_inset.set_xticks(x)
    ax_inset.set_xticklabels(systems, rotation=45, fontsize=7, ha='right')
    ax_inset.set_ylabel("τ_eff [ns]", fontsize=8)
    ax_inset.tick_params(axis='y', labelsize=7)
    ax_inset.set_title("τ", fontsize=9)

# ── Layout ────────────────────────────────────────────────────────────
plt.tight_layout()
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ── Files and systems ───────────────────────────────────────────────
datasets = [
    ('corr_results/9res_corr_results_spacers.pkl',
     ['2_close','2_spread','3_close','3_spread'],
     '9res (spacers)'),

    ('corr_results/9res_corr_results_2.pkl',
     ['equal', 'aromatic', 'nonpolar', 'polar', 'charged'],
     '9res'),

    ('corr_results/3res_corr_results_2.pkl',
     ['equal', 'aromatic', 'nonpolar', 'polar', 'charged'],
     '3res'),
]

# Colors
clrs = ['black','red','blue','green','purple','orange']

plt.rcParams.update({"font.size": 14})

# ── Create subplot grid (3 rows × 2 cols) ───────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(12, 12))

# ── Loop ───────────────────────────────────────────────────────────
for row, (save_path, systems, title_row) in enumerate(datasets):

    with open(save_path, 'rb') as f:
        results = pickle.load(f)

    ax_corr = axes[row, 0]
    ax_bar  = axes[row, 1]

    tau_vals = []

    # ── Correlation plots ──────────────────────────────────────────
    for i, label in enumerate(systems):
        r = results[label]

        ax_corr.plot(r['tM'], r['corrN'], '.', color=clrs[i], alpha=0.5)
        ax_corr.plot(r['tM'], r['corrFit'], '-', color=clrs[i],
                     label=f"{label} (τ={r['t_eff']:.2f})")

        tau_vals.append(r['t_eff'])

    ax_corr.set_title(f"{title_row} — Correlation")
    ax_corr.set_xlabel("t [ns]")
    ax_corr.set_ylabel("C(t)")
    ax_corr.legend(frameon=False, fontsize=10)

    # ── Bar plot ───────────────────────────────────────────────────
    x = np.arange(len(systems))
    ax_bar.bar(x, tau_vals, color=clrs[:len(systems)])

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(systems, rotation=45)
    ax_bar.set_ylabel("τ_eff [ns]")
    ax_bar.set_title(f"{title_row} — τ comparison")

# ── Layout ─────────────────────────────────────────────────────────
plt.tight_layout()
plt.show()

### Water Bridge analysis

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ── Config ─────────────────────────────────────────────────────────────────────
pkl_files = [
    ('./corr_results/water_corr_resultsi_9res.pkl',   'Dataset 1 (9res)'),
    ('./corr_results/water_corr_resultsi_9res_2.pkl', 'Dataset 2 (9res_2)'),
    ('./corr_results/water_corr_resultsi_3res.pkl',   'Dataset 3 (3res)'),
]

clrs  = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2 = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

row_titles = ['9res', '9res_2', '3res']

# ── Figure: 3 rows × 2 cols ────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 15))

for row, (pkl_path, _) in enumerate(pkl_files):
    with open(pkl_path, 'rb') as f:
        results = pickle.load(f)

    ax_in  = axes[row, 0]
    ax_out = axes[row, 1]

    # Row label on the left axis
    ax_in.set_title(f'Inner Layer — {row_titles[row]}', fontsize=12, fontweight='bold')
    ax_out.set_title(f'Outer Layer — {row_titles[row]}', fontsize=12, fontweight='bold')

    for idx, (label, res) in enumerate(results.items()):
        tM = res['tM']

        # ── inner ──────────────────────────────────────────────────────────────
        ax_in.plot(tM, res['corrN'],   color=clrs[idx], lw=1.8, label=label)
        ax_in.plot(tM, res['corrFit'], color=clrs[idx], lw=1.5, ls='--',
                   label=f"{label} fit (τ_eff={res['t_eff']:.3f} ns)")

        # ── outer ──────────────────────────────────────────────────────────────
        ax_out.fill_between(tM,
                            res['corrN_2'] - res['corrN_std_2'],
                            res['corrN_2'] + res['corrN_std_2'],
                            color=clrs2[idx], alpha=0.15)
        ax_out.plot(tM, res['corrN_2'],   color=clrs2[idx], lw=1.8, label=label)
        ax_out.plot(tM, res['corrFit_2'], color=clrs2[idx], lw=1.5, ls='--',
                    label=f"{label} fit (τ_eff={res['t_eff_2']:.3f} ns)")

    for ax in (ax_in, ax_out):
        ax.set_xlabel('Time (ns)', fontsize=11)
        ax.set_ylabel('C(t)',      fontsize=11)
        ax.set_xlim(tM[0], tM[-1])
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.tick_params(labelsize=9)

plt.suptitle('Water–Protein Contact Correlation', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()

save_path = './corr_results/water_corr_comparison_3x2.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved: {save_path}")
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ── Config ─────────────────────────────────────────────────────────────────────
pkl_files = [
    ('./corr_results/water_corr_resultsi_3res.pkl',   'Dataset 1 (3res)'),
    ('./corr_results/water_corr_resultsi_9res.pkl',   'Dataset 2 (9res)'),
    ('./corr_results/water_corr_resultsi_9res_2.pkl', 'Dataset 3 (9res_2)'),

]

clrs  = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2 = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

row_titles = [ '3res','9res', '9res_2']

# ── Figure: 3 rows × 2 cols ────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 15))

for row, (pkl_path, _) in enumerate(pkl_files):
    with open(pkl_path, 'rb') as f:
        results = pickle.load(f)

    ax_in  = axes[row, 0]
    ax_out = axes[row, 1]

    # Titles
    ax_in.set_title(f'Inner Layer — {row_titles[row]}', fontsize=12, fontweight='bold')
    ax_out.set_title(f'Outer Layer — {row_titles[row]}', fontsize=12, fontweight='bold')

    # ── Plot curves ────────────────────────────────────────────────────────────
    for idx, (label, res) in enumerate(results.items()):
        tM = res['tM']

        # Inner
        ax_in.plot(tM, res['corrN'],   color=clrs[idx], lw=1.8, label=label)
        ax_in.plot(tM, res['corrFit'], color=clrs[idx], lw=1.5, ls='--',
                   label=f"{label} fit (τ={res['t_eff']:.2f} ns)")

        # Outer
        ax_out.fill_between(
            tM,
            res['corrN_2'] - res['corrN_std_2'],
            res['corrN_2'] + res['corrN_std_2'],
            color=clrs2[idx], alpha=0.15
        )
        ax_out.plot(tM, res['corrN_2'],   color=clrs2[idx], lw=1.8, label=label)
        ax_out.plot(tM, res['corrFit_2'], color=clrs2[idx], lw=1.5, ls='--',
                    label=f"{label} fit (τ={res['t_eff_2']:.2f} ns)")

    # ── Axes styling ───────────────────────────────────────────────────────────
    for ax in (ax_in, ax_out):
        ax.set_xlabel('Time (ns)', fontsize=11)
        ax.set_ylabel('C(t)', fontsize=11)
        ax.set_xlim(tM[0], tM[-1])
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.tick_params(labelsize=9)

    # ── τ inset bar plots ──────────────────────────────────────────────────────
    labels = list(results.keys())
    tau_inner = [results[k]['t_eff'] for k in labels]
    tau_outer = [results[k]['t_eff_2'] for k in labels]

    x = np.arange(len(labels))

    # Inner inset
    axins_in = inset_axes(ax_in, width="45%", height="45%", loc='upper center')
    axins_in.bar(x, tau_inner, color=clrs[:len(labels)], alpha=0.9)
    #axins_in.set_title("τ (ns)", fontsize=8)
    axins_in.set_xticks([])
    axins_in.tick_params(axis='y', labelsize=7)
    axins_in.grid(True, alpha=0.2)

    # Optional value labels
    for i, v in enumerate(tau_inner):
        axins_in.text(i, v, f"{v:.2f}", ha='center', va='bottom', fontsize=6)

    # Outer inset
    axins_out = inset_axes(ax_out, width="45%", height="45%", loc='upper center')
    axins_out.bar(x, tau_outer, color=clrs2[:len(labels)], alpha=0.9)
    #axins_out.set_title("τ (ns)", fontsize=8)
    axins_out.set_xticks([])
    axins_out.tick_params(axis='y', labelsize=7)
    axins_out.grid(True, alpha=0.2)

    for i, v in enumerate(tau_outer):
        axins_out.text(i, v, f"{v:.2f}", ha='center', va='bottom', fontsize=6)

# ── Final layout ───────────────────────────────────────────────────────────────
plt.suptitle('Water–Protein Contact Correlation', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()

save_path = './corr_results/water_corr_comparison_3x2.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved: {save_path}")

plt.show()

In [ ]:
df = pd.read_csv('data/equal_waterbridges_9res.csv')
df.tail()


In [ ]:
import pandas as pd

total_ratio_df = pd.DataFrame()

# -------------------------
# 3res / 9res systems
# -------------------------
for i in ['3res','9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        # Load the file
        df = pd.read_csv(f'data/{j}_waterbridges_{i}.csv')
        
        # Create temporary DataFrame with proper column name
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{j[0].upper()}'] = df['WBC_perresidue'].values
        
        # Append to total DataFrame
        total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)

# -------------------------
# close / spread systems
# -------------------------
for i in ['2_close','2_spread','3_close','3_spread']:
    
    if i == '2_close':
        name = 'GGGXGXGGG'
        num_stickers = 2
    elif i == '2_spread':
        name = 'GXGGGGGXG'
        num_stickers = 2
    elif i == '3_close':
        name = 'GGGXXXGGG'
        num_stickers = 3
    else:
        name = 'GXGGXGGXG'
        num_stickers = 3

    num_stickers = 1
    # Use the correct filename for each system
    df = pd.read_csv(f'data/{i}_waterbridges_9res.csv')
    
    temp_df = pd.DataFrame()
    # Normalize by number of stickers if needed
    temp_df[name] = df['WBC_perresidue'].values / num_stickers
    
    total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)

# -------------------------
# Rename columns (to match size_df)
# -------------------------
total_ratio_df = total_ratio_df.rename(columns={
    '3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P', '3N':'3N',
    '9E':'9E', '9C':'9C', '9A':'9A', '9P':'9P', '9N':'9N',
    'GGGXGXGGG':'X2', 'GXGGGGGXG':'X2S',
    'GGGXXXGGG':'X3', 'GXGGXGGXG':'X3S'
})

total_ratio_df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Means and standard deviations
means = total_ratio_df.mean()
stds  = total_ratio_df.std()

# X positions
x = np.arange(len(means))
width = 0.6

# Plot
plt.figure(figsize=(14,6))
plt.bar(x, means, width, yerr=stds, capsize=5, color='mediumseagreen', alpha=0.8)

# Labels
plt.xticks(x, means.index, rotation=45, ha='right')
plt.ylabel('WaterBridge Count')
plt.xlabel('System')
plt.title('Comparison of Water Bridges Across Systems')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Water:Peptide ratios

In [ ]:
import pandas as pd

total_ratio_df = pd.DataFrame()

# -------------------------
# 3res / 9res systems
# -------------------------
for i in ['3res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{j}_cluster_water.csv')  # adjust if needed
        
        # compute ratio
        ratio = df['water_mass'] / df['protein_mass']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{j[0].upper()}'] = ratio.values
        
        total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)
        
        
for i in ['9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{j}_cluster_water_9res.csv')  # adjust if needed
        
        # compute ratio
        ratio = df['water_mass'] / df['protein_mass']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{j[0].upper()}'] = ratio.values
        
        total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)


# -------------------------
# close / spread systems
# -------------------------
for i in ['2_close','2_spread','3_close','3_spread']:
    
    if i == '2_close':
        name = 'GGGXGXGGG'
        num_stickers = 2
    elif i == '2_spread':
        name = 'GXGGGGGXG'
        num_stickers = 2
    elif i == '3_close':
        name = 'GGGXXXGGG'
        num_stickers = 3
    else:
        name = 'GXGGXGGXG'
        num_stickers = 3
        
        
    num_stickers = 1

    df = pd.read_csv(f'data/{i}_cluster_water_9res.csv')
    
    ratio = df['water_mass'] / df['protein_mass']
    
    temp_df = pd.DataFrame()
    temp_df[name] = ratio.values / num_stickers   # normalize like size
    
    total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)


# -------------------------
# Rename columns (same as size)
# -------------------------
total_ratio_df = total_ratio_df.rename(columns={
    '3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P', '3N':'3N',
    '9E':'9E', '9C':'9C', '9A':'9A', '9P':'9P', '9N':'9N',
    'GGGXGXGGG':'X2', 'GXGGGGGXG':'X2S',
    'GGGXXXGGG':'X3', 'GXGGXGGXG':'X3S'
})

In [ ]:
total_ratio_df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Compute statistics
ratio_mean = total_ratio_df.mean()
ratio_std  = total_ratio_df.std()

# X positions
x = np.arange(len(ratio_mean))

# Plot
plt.figure(figsize=(10,6))

plt.bar(x, ratio_mean, yerr=ratio_std, capsize=5)

# Labels
plt.xticks(x, ratio_mean.index)
plt.ylabel('Water / Protein Mass Ratio')
plt.xlabel('System')
plt.title('Hydration Level Across Systems')
plt.ylim(0.6,1.2)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

total_layer_ratio_df = pd.DataFrame()

# -------------------------
# 3res / 9res systems
# -------------------------
for i in ['3res']:
    for sy in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{sy}_inner_outer_water.csv')
        
        # Compute ratios
        inner_ratio = df['inner_water_unique'] / df['total_water']
        outer_ratio = df['outer_water'] / df['total_water']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{sy[0].upper()}_in']  = inner_ratio.values
        temp_df[f'{i[0]}{sy[0].upper()}_out'] = outer_ratio.values
        
        total_layer_ratio_df = pd.concat([total_layer_ratio_df, temp_df], axis=1)
        
        
for i in ['9res']:
    for sy in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{sy}_inner_outer_water_{i}.csv')
        
        # Compute ratios
        inner_ratio = df['inner_water_unique'] / df['total_water']
        outer_ratio = df['outer_water'] / df['total_water']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{sy[0].upper()}_in']  = inner_ratio.values
        temp_df[f'{i[0]}{sy[0].upper()}_out'] = outer_ratio.values
        
        total_layer_ratio_df = pd.concat([total_layer_ratio_df, temp_df], axis=1)


# -------------------------
# close / spread systems
# -------------------------
for i in ['2_close','2_spread','3_close','3_spread']:
    
    if i == '2_close':
        name = 'X2'
    elif i == '2_spread':
        name = 'X2S'
    elif i == '3_close':
        name = 'X3'
    else:
        name = 'X3S'
    
    df = pd.read_csv(f'data/{i}_inner_outer_water_9res.csv')
    
    inner_ratio = df['inner_water_unique'] / df['total_water']
    outer_ratio = df['outer_water'] / df['total_water']
    
    temp_df = pd.DataFrame()
    temp_df[f'{name}_in']  = inner_ratio.values
    temp_df[f'{name}_out'] = outer_ratio.values
    
    total_layer_ratio_df = pd.concat([total_layer_ratio_df, temp_df], axis=1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

inner_cols = [col for col in total_layer_ratio_df.columns if col.endswith('_in')]
outer_cols = [col for col in total_layer_ratio_df.columns if col.endswith('_out')]

labels = [col.replace('_in','') for col in inner_cols]

inner_mean = total_layer_ratio_df[inner_cols].mean()
inner_std  = total_layer_ratio_df[inner_cols].std()

outer_mean = total_layer_ratio_df[outer_cols].mean()
outer_std  = total_layer_ratio_df[outer_cols].std()

x = np.arange(len(labels))
width = 0.35

fig, ax1 = plt.subplots(figsize=(12,6))
ax2 = ax1.twinx()

# INNER (solid bars, left axis)
bars1 = ax1.bar(
    x - width/2,
    inner_mean,
    width,
    yerr=inner_std,
    capsize=5,
    label='Inner',
    alpha=0.9
)

# OUTER (lighter bars, right axis)
bars2 = ax2.bar(
    x + width/2,
    outer_mean,
    width,
    yerr=outer_std,
    capsize=5,
    label='Outer',
    alpha=0.4
)

# Labels
ax1.set_ylabel('Inner Water Fraction')
ax2.set_ylabel('Outer Water Fraction')
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_xlabel('System')

ax2.set_ylim(0.8,1)

plt.title('Inner vs Outer Water Distribution')

# Combined legend
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Columns
inner_cols = [col for col in total_layer_ratio_df.columns if col.endswith('_in')]
outer_cols = [col for col in total_layer_ratio_df.columns if col.endswith('_out')]

labels = [col.replace('_in','') for col in inner_cols]

# Means and std
inner_mean = total_layer_ratio_df[inner_cols].mean()
inner_std  = total_layer_ratio_df[inner_cols].std()
outer_mean = total_layer_ratio_df[outer_cols].mean()
outer_std  = total_layer_ratio_df[outer_cols].std()

x = np.arange(len(labels))
width = 0.6

fig, axes = plt.subplots(2, 1, figsize=(14,10), 
                         sharex=False)

# ----------------------
# Top: Inner water
# ----------------------
axes[0].bar(x, inner_mean, width, yerr=inner_std, capsize=5, color='skyblue')
axes[0].set_ylabel('Inner Water Fraction', fontsize=12)
axes[0].set_title('Inner Water Distribution', fontsize=14)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=45, ha='right')

# ----------------------
# Bottom: Outer water
# ----------------------
axes[1].bar(x, outer_mean, width, yerr=outer_std, capsize=5, color='salmon')
axes[1].set_ylabel('Outer Water Fraction', fontsize=12)
axes[1].set_xlabel('System', fontsize=12)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha='right')
axes[1].set_title('Outer Water Distribution', fontsize=14)
axes[1].grid(axis='y', alpha=0.3)

axes[1].set_ylim(0.8,1)

plt.tight_layout()
plt.show()

In [ ]:
pd.read_csv('data/aromatic_cluster_water.csv').head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

systems = ['equal','charged','aromatic','polar','nonpolar']
#systems = ['equal','charged','aromatic','polar','nonpolar','2_close','2_spread','3_close','3_spread']
all_data = []

for sy in systems:
    
    df = pd.read_csv(f'data/{sy}_cluster_water.csv')  # your file name
    
    # Compute protein-to-water mass ratio
    df['prot_water_ratio'] = df['water_mass'] / df['protein_mass']
    df['system'] = sy
    all_data.append(df)

# Combine all systems
combined_df = pd.concat(all_data, ignore_index=True)

# Compute mean values per system
mean_df = combined_df.groupby('system')[['protein_mass','water_mass','prot_water_ratio']].mean().reset_index()

# Subplot: two rows
x = range(len(mean_df))
fig, axes = plt.subplots(2, 1, figsize=(8,10), sharex=True)

# --- Top row: Protein Mass ---
axes[0].bar(x, mean_df['protein_mass'], color='lightgreen')
axes[0].set_ylabel('Mean Protein Mass')
axes[0].set_title('Protein Mass Across Systems')

# --- Bottom row: Water Mass / Protein-to-Water Ratio ---
axes[1].bar(x, mean_df['prot_water_ratio'], color='orange')
axes[1].set_ylabel('Mean Water/Protein Mass Ratio')
axes[1].set_title('Protein/Water Mass Ratio Across Systems')
axes[1].set_xticks(x)
axes[1].set_xticklabels(mean_df['system'])
#axes[1].set_ylim(0.6,0.8)
plt.tight_layout()
plt.show()

### Count water 

In [ ]:
pd.read_csv('data/aromatic_inner_outer_water.csv').head()

In [ ]:

systems = ['equal','charged','aromatic','polar','nonpolar']
systems = ['equal','charged','aromatic','polar','nonpolar','2_close','2_spread','3_close','3_spread']

all_data = []

for sy in systems:
    df = pd.read_csv(f'data/{sy}_inner_outer_water_9res.csv')
    
    # Compute ratios
    df['inner_ratio'] = df['inner_water_unique'] / df['total_water']
    df['outer_ratio'] = df['outer_water'] / df['total_water']
    
    # Add a column for the system name
    df['system'] = sy
    
    all_data.append(df)

# Combine all systems into a single DataFrame
combined_df = pd.concat(all_data, ignore_index=True)

# Save to CSV if needed
#combined_df.to_csv('data/combined_inner_outer_ratios.csv', index=False)
combined_df.head()

In [ ]:

import matplotlib.pyplot as plt

# Compute mean ratios per system
mean_df = combined_df.groupby('system')[['inner_ratio','outer_ratio']].mean().reset_index()

systems = mean_df['system']
inner = mean_df['inner_ratio']
outer = mean_df['outer_ratio']

x = range(len(systems))

fig, axes = plt.subplots(2, 1, figsize=(8, 10), sharex=True)

# --- Top row: Inner water ---
axes[0].bar(x, inner, color='skyblue')
axes[0].set_ylabel('Mean Inner Water Fraction')
axes[0].set_title('Inner Water Fraction Across Systems')
axes[0].set_ylim(0.01,0.15)

# --- Bottom row: Outer water ---
axes[1].bar(x, outer, color='salmon')
axes[1].set_ylabel('Mean Outer Water Fraction')
axes[1].set_title('Outer Water Fraction Across Systems')
axes[1].set_xticks(x)
axes[1].set_xticklabels(systems)
axes[1].set_ylim(0.8,1)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

total_layer_ratio_df = pd.DataFrame()

# -------------------------
# 3res / 9res systems
# -------------------------
for i in ['3res']:
    for sy in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{sy}_inner_outer_water.csv')
        
        # Compute ratios
        inner_ratio = df['inner_water_unique'] / df['total_water']
        outer_ratio = df['outer_water'] / df['total_water']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{sy[0].upper()}_in']  = inner_ratio.values
        temp_df[f'{i[0]}{sy[0].upper()}_out'] = outer_ratio.values
        
        total_layer_ratio_df = pd.concat([total_layer_ratio_df, temp_df], axis=1)

for i in ['9res']:
    for sy in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{sy}_inner_outer_water_{i}.csv')
        
        # Compute ratios
        inner_ratio = df['inner_water_unique'] / df['total_water']
        outer_ratio = df['outer_water'] / df['total_water']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{sy[0].upper()}_in']  = inner_ratio.values
        temp_df[f'{i[0]}{sy[0].upper()}_out'] = outer_ratio.values
        
        total_layer_ratio_df = pd.concat([total_layer_ratio_df, temp_df], axis=1)


# -------------------------
# close / spread systems
# -------------------------
for i in ['2_close','2_spread','3_close','3_spread']:
    
    if i == '2_close':
        name = 'X2'
    elif i == '2_spread':
        name = 'X2S'
    elif i == '3_close':
        name = 'X3'
    else:
        name = 'X3S'
    
    df = pd.read_csv(f'data/{i}_inner_outer_water_9res.csv')
    
    inner_ratio = df['inner_water_unique'] / df['total_water']
    outer_ratio = df['outer_water'] / df['total_water']
    
    temp_df = pd.DataFrame()
    temp_df[f'{name}_in']  = inner_ratio.values
    temp_df[f'{name}_out'] = outer_ratio.values
    
    total_layer_ratio_df = pd.concat([total_layer_ratio_df, temp_df], axis=1)

In [ ]:
total_layer_ratio_df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Columns
inner_cols = [col for col in total_layer_ratio_df.columns if col.endswith('_in')]
outer_cols = [col for col in total_layer_ratio_df.columns if col.endswith('_out')]

labels = [col.replace('_in','') for col in inner_cols]

# Means and std
inner_mean = total_layer_ratio_df[inner_cols].mean()
inner_std  = total_layer_ratio_df[inner_cols].std()
outer_mean = total_layer_ratio_df[outer_cols].mean()
outer_std  = total_layer_ratio_df[outer_cols].std()

x = np.arange(len(labels))
width = 0.6

fig, axes = plt.subplots(2, 1, figsize=(14,10), sharex=True)

# ----------------------
# Top: Inner water
# ----------------------
axes[0].bar(x, inner_mean, width, yerr=inner_std, capsize=5, color='skyblue')
axes[0].set_ylabel('Inner Water Fraction', fontsize=12)
axes[0].set_title('Inner Water Distribution', fontsize=14)
axes[0].grid(axis='y', alpha=0.3)

# ----------------------
# Bottom: Outer water
# ----------------------
axes[1].bar(x, outer_mean, width, yerr=outer_std, capsize=5, color='salmon')
axes[1].set_ylabel('Outer Water Fraction', fontsize=12)
axes[1].set_xlabel('System', fontsize=12)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha='right')
axes[1].set_title('Outer Water Distribution', fontsize=14)
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(0.85,1)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Define systems
systems_3res = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
systems_9res = systems_3res
systems_spacers = ['2_close','2_spread','3_close','3_spread']

layers = ['Inner','Outer']

# Function to compute Teff
def compute_teff(df):
    return {lay: fit_constrained(df[lay].values, lay) for lay in layers}

# ---- Collect Teff for all systems ----
teff_3res = {}
for sys in systems_3res:
    df = pd.read_csv(f'data/sp_2res_{sys}_new.csv')
    teff_3res[sys] = compute_teff(df)

teff_9res = {}
for sys in systems_9res:
    df = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
    teff_9res[sys] = compute_teff(df)

teff_spacers = {}
spacer_map = {
    '2_close':  'X2',
    '2_spread': 'X2S',
    '3_close':  'X3',
    '3_spread': 'X3S'
}
for sys in systems_spacers:
    df = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
    teff_spacers[sys] = compute_teff(df)

# ---- Plot comparison ----
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 12))

# 1. 3-res systems
ax = axes[0]
for i, sys in enumerate(systems_3res):
    ax.bar([x + i*0.15 for x in range(2)], list(teff_3res[sys].values()), width=0.15, label=sys)
ax.set_xticks(range(2))
ax.set_xticklabels(layers)
ax.set_title("3-res Systems")
ax.legend()

# 2. 9-res systems
ax = axes[1]
for i, sys in enumerate(systems_9res):
    ax.bar([x + i*0.15 for x in range(2)], list(teff_9res[sys].values()), width=0.15, label=sys)
ax.set_xticks(range(2))
ax.set_xticklabels(layers)
ax.set_title("9-res Systems")
ax.legend()

# 3. Spacer systems
ax = axes[2]
for i, sys in enumerate(systems_spacers):
    ax.bar([x + i*0.15 for x in range(2)], list(teff_spacers[sys].values()), width=0.15, label=sys)
ax.set_xticks(range(2))
ax.set_xticklabels(layers)
ax.set_title("Spacer Systems")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
x = [1,2,4,5,6,7]
x[-2:0]

In [ ]:
total_size_df = pd.DataFrame()

for i in ['3res','9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/size_data/{i}/{j}_size.csv', 
                              index_col=0)
        temp_df_2 = pd.DataFrame()
        temp_df_2[f'{i[0]}{j[0].upper()}'] = temp_df['prot-size'].values
        total_size_df = pd.concat([total_size_df, temp_df_2], axis=1)
        
for i in ['2_close','2_spread','3_close','3_spread']:
    if i == '2_close':
        name = 'GGGXGXGGG'
        num_stickers = 2
    elif i == '2_spread':
        name = 'GXGGGGGXG'
        num_stickers = 2
    elif i == '3_close':
        name = 'GGGXXXGGG'
        num_stickers = 3
    else:
        name = 'GXGGXGGXG'
        num_stickers = 3
        
        
    #num_stickers = 1
    temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/size_data/close_spread/{i}_size.csv',
                          index_col=0)
    temp_df_2 = pd.DataFrame()
    temp_df_2[f'{name}'] = temp_df['prot-size'].values / num_stickers
    total_size_df = pd.concat([total_size_df, temp_df_2], axis=1)


total_size_df

total_size_df = total_size_df.rename(columns={'3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P',
       '3N':'3N', '9E':'9E', '9C':'9C', '9A':'9A',
       '9P':'9P', '9N':'9N', 'GGGXGXGGG':'X2','GXGGGGGXG':'X2S', 'GGGXXXGGG':'X3',
       'GXGGXGGXG':'X3S'})


In [ ]:
total_size_df

In [ ]:
total_sasa_df = pd.DataFrame()

# -------------------------------
# 3res and 9res systems
# -------------------------------
systems = ['equal','charged','aromatic','polar','nonpolar']

df_3res = pd.read_csv('data/sasa_persidue_3res.csv')
df_9res = pd.read_csv('data/sasa_persidue_9res.csv')

for j in systems:
    temp_df = pd.DataFrame()
    temp_df[f'3{j[0].upper()}'] = df_3res[j].values
    total_sasa_df = pd.concat([total_sasa_df, temp_df], axis=1)

for j in systems:
    temp_df = pd.DataFrame()
    temp_df[f'9{j[0].upper()}'] = df_9res[j].values
    total_sasa_df = pd.concat([total_sasa_df, temp_df], axis=1)

# -------------------------------
# Spacer systems (9res_spacers)
# -------------------------------
df_9res_sp = pd.read_csv('data/sasa_persidue_9res_spacers.csv')

spacer_map = {
    '2_close':  ('X2', 2),
    '2_spread': ('X2S', 2),
    '3_close':  ('X3', 3),
    '3_spread': ('X3S', 3)
}

for key, (label, n_stickers) in spacer_map.items():
    temp_df = pd.DataFrame()
    
    # assuming column names in df_9res_sp match keys
    temp_df[label] = df_9res_sp[key].values
    
    total_sasa_df = pd.concat([total_sasa_df, temp_df], axis=1)

# -------------------------------
# Final dataframe
total_sasa_df.head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# Data
# -------------------------------
systems = total_size_df.columns
x = np.arange(len(systems))

# Means
size_mean = total_size_df.mean()
sasa_mean = total_sasa_df.mean()

# -------------------------------
# Plot
# -------------------------------
fig, ax1 = plt.subplots(figsize=(10,6))

# --- Cluster size (bars, LEFT axis) ---
#ax1.bar(x, size_mean.values)
ax1.plot(x,size_mean.values,'o',color='black')
ax1.set_ylabel('Cluster Size')
ax1.set_xticks(x)
ax1.set_xticklabels(systems, rotation=45)

# --- SASA (points, RIGHT axis) ---
ax2 = ax1.twinx()
ax2.plot(x, sasa_mean.values, 'o',color = 'red')
ax2.set_ylabel('Average SASA')

# -------------------------------
# Colors (2 types)
# -------------------------------
ax1.set_ylabel('Cluster Size', color='blue')
ax2.set_ylabel('Average SASA', color='red')

ax1.tick_params(axis='y', colors='blue')
ax2.tick_params(axis='y', colors='red')

# -------------------------------
# Final
# -------------------------------
plt.title('Cluster Size vs SASA (Mean)')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# Data
# -------------------------------
systems = total_size_df.columns
x = np.arange(len(systems))

size_mean = total_size_df.mean()
sasa_mean = total_sasa_df.mean()

# -------------------------------
# Subplots (shared x-axis)
# -------------------------------
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10,8), sharex=True)

# -------------------------------
# Top: Cluster Size
# -------------------------------
ax1.bar(x, size_mean.values)
ax1.set_ylabel('Cluster Size')
ax1.set_title('Cluster Size vs SASA')
ax1.grid(alpha=0.3)
ax1.set_ylim(0.6,1)

# -------------------------------
# Bottom: SASA
# -------------------------------
ax2.bar(x, sasa_mean.values)
ax2.set_ylabel('Average SASA')
ax2.set_xticks(x)
ax2.set_xticklabels(systems, rotation=45)
ax2.grid(alpha=0.3)

# -------------------------------
# Layout
# -------------------------------
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Means
size_mean = total_size_df.mean()
sasa_mean = total_sasa_df.mean()

# -------------------------------
# Scatter plot
# -------------------------------
plt.figure(figsize=(7,6))

plt.scatter(size_mean.values, sasa_mean.values)

# Label each point
for i, system in enumerate(size_mean.index):
    plt.text(size_mean.values[i], sasa_mean.values[i], system)

# -------------------------------
# Labels
# -------------------------------
plt.xlabel('Cluster Size')
plt.ylabel('Average SASA')
plt.title('Cluster Size vs SASA')

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Define systems
systems_3res = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
systems_9res = systems_3res
systems_spacers = ['2_close','2_spread','3_close','3_spread']

layers = ['Inner','Mid','Outer']

# Function to compute Teff
def compute_teff(df):
    return {lay: fit_constrained(df[lay].values, lay) for lay in layers}

# ---- Collect Teff for all systems ----
teff_3res = {}
for sys in systems_3res:
    df = pd.read_csv(f'data/sp_3res_{sys}.csv')
    teff_3res[sys] = compute_teff(df)

teff_9res = {}
for sys in systems_9res:
    df = pd.read_csv(f'data/sp_9res_{sys}.csv')
    teff_9res[sys] = compute_teff(df)

teff_spacers = {}
spacer_map = {
    '2_close':  'X2',
    '2_spread': 'X2S',
    '3_close':  'X3',
    '3_spread': 'X3S'
}
for sys in systems_spacers:
    df = pd.read_csv(f'data/sp_9res_{sys}.csv')
    teff_spacers[sys] = compute_teff(df)

# ---- Plot comparison ----
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 12))

# 1. 3-res systems
ax = axes[0]
for i, sys in enumerate(systems_3res):
    ax.bar([x + i*0.15 for x in range(3)], list(teff_3res[sys].values()), width=0.15, label=sys)
ax.set_xticks(range(3))
ax.set_xticklabels(layers)
ax.set_title("3-res Systems")
ax.legend()

# 2. 9-res systems
ax = axes[1]
for i, sys in enumerate(systems_9res):
    ax.bar([x + i*0.15 for x in range(3)], list(teff_9res[sys].values()), width=0.15, label=sys)
ax.set_xticks(range(3))
ax.set_xticklabels(layers)
ax.set_title("9-res Systems")
ax.legend()

# 3. Spacer systems
ax = axes[2]
for i, sys in enumerate(systems_spacers):
    ax.bar([x + i*0.15 for x in range(3)], list(teff_spacers[sys].values()), width=0.15, label=sys)
ax.set_xticks(range(3))
ax.set_xticklabels(layers)
ax.set_title("Spacer Systems")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Define systems
systems_3res = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
systems_9res = systems_3res
systems_spacers = ['2_close','2_spread','3_close','3_spread']

layers = ['Inner','Outer']

# Function to compute Teff
def compute_teff(df):
    return {lay: fit_constrained(df[lay].values, lay) for lay in layers}

# ---- Collect Teff for all systems ----
teff_3res = {}
for sys in systems_3res:
    df = pd.read_csv(f'data/sp_2res_{sys}_new.csv')
    teff_3res[sys] = compute_teff(df)

teff_9res = {}
for sys in systems_9res:
    df = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
    teff_9res[sys] = compute_teff(df)

teff_spacers = {}
spacer_map = {
    '2_close':  'X2',
    '2_spread': 'X2S',
    '3_close':  'X3',
    '3_spread': 'X3S'
}
for sys in systems_spacers:
    df = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
    teff_spacers[sys] = compute_teff(df)

# ---- Plot comparison ----
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 12))

# 1. 3-res systems
ax = axes[0]
for i, sys in enumerate(systems_3res):
    ax.bar([x + i*0.15 for x in range(2)], list(teff_3res[sys].values()), width=0.15, label=sys)
ax.set_xticks(range(2))
ax.set_xticklabels(layers)
ax.set_title("3-res Systems")
ax.legend()

# 2. 9-res systems
ax = axes[1]
for i, sys in enumerate(systems_9res):
    ax.bar([x + i*0.15 for x in range(2)], list(teff_9res[sys].values()), width=0.15, label=sys)
ax.set_xticks(range(2))
ax.set_xticklabels(layers)
ax.set_title("9-res Systems")
ax.legend()

# 3. Spacer systems
ax = axes[2]
for i, sys in enumerate(systems_spacers):
    ax.bar([x + i*0.15 for x in range(2)], list(teff_spacers[sys].values()), width=0.15, label=sys)
ax.set_xticks(range(2))
ax.set_xticklabels(layers)
ax.set_title("Spacer Systems")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 3. Main Comparison Loop
systems = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
#systems = ['2_close', '2_spread', '3_close', '3_spread']
results = []

layers = ['Inner','Outer']


for sys in systems:
    plt.figure(figsize=(8, 5))
    
    # Load data
    try:
        loaded_df = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
    except FileNotFoundError:
        print(f"File for {sys} not found, skipping...")
        continue
        
    # Calculate Teff for each layer and plot
    teff_inner = fit_constrained(loaded_df['Inner'].values, 'Inner')
    #teff_mid   = fit_constrained(loaded_df['Mid'].values, 'Mid')
    teff_outer = fit_constrained(loaded_df['Outer'].values, 'Outer')

    # Store results for the final summary table
    results.append({
        'System': sys,
        'Inner_Teff': teff_inner,
        #'Mid_Teff': teff_mid,
        'Outer_Teff': teff_outer
    })

    plt.xlabel('Time')
    plt.ylabel('Normalized SP')
    plt.title(f'Constrained Double Exponential Fit - System: {sys.capitalize()}')
    plt.legend()
    plt.show()

# 4. Final Comparison Table
summary_df = pd.DataFrame(results)
print("\n--- Final Teff Comparison Summary ---")
print(summary_df.to_string(index=False))

# Optional: Plot the comparison of Teff across systems
summary_df.set_index('System').plot(kind='bar', figsize=(10, 6))
plt.ylabel('Effective Mean Residence Time (Teff)')
plt.title('Comparison of Teff across Chemical Systems and Layers')
plt.xticks(rotation=45)
plt.show()

In [ ]:

import seaborn as sns

# Assuming 'summary_df' was created in the previous step
# Melt the dataframe for easier plotting with Seaborn
melted_df = summary_df.melt(id_vars='System', var_name='Layer', value_name='Teff')

# Define the layers to plot separately
layers = ['Inner_Teff', 'Outer_Teff']

for layer_name in layers:
    plt.figure(figsize=(9, 5))
    
    # Filter data for the specific layer
    layer_data = summary_df[['System', layer_name]].sort_values(by=layer_name, ascending=False)
    
    # Create bar plot
    sns.barplot(x='System', y=layer_name, data=layer_data, palette='viridis')
    
    plt.title(f'Comparison of $T_{{eff}}$ across Systems: {layer_name.split("_")[0]} Layer')
    plt.ylabel('Mean Residence Time ($T_{eff}$)')
    plt.xlabel('Chemical System')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

# Print the numeric breakdown
for layer in layers:
    print(f"\n--- Statistics for {layer} ---")
    print(summary_df[['System', layer]].sort_values(by=layer, ascending=False).to_string(index=False))
    